In [0]:
# Replace with your actual access key from Azure portal > Storage account > Access keys
spark.conf.set(
  "fs.azure.account.key.subramani21.blob.core.windows.net",
  "FGOU2xJ7nBFGsKe8YW2VhNQxWWYmTF6Ez+Q5szV1x3V4BOW/ey9f+D1f4AQ8l1kwahHknbp8rv6e+AStTE4HyQ=="
)


In [0]:
# Load orders.csv (rename file if needed to remove spaces)
orders_df = spark.read.option("header", "true").option("inferSchema", "true").csv(
  "wasbs://images@subramani21.blob.core.windows.net/orders (4).csv"
)

# Load customers.csv
customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv(
  "wasbs://images@subramani21.blob.core.windows.net/customers (3).csv"
)

# Preview both
print("📦 Orders Data")
orders_df.show(5)

print("📦 Customers Data")
customers_df.show(5)


📦 Orders Data
+--------+-----------+----------+-------------+-------------+------------+
|order_id|customer_id|order_date|expected_date|delivery_date|     product|
+--------+-----------+----------+-------------+-------------+------------+
|       1|         90|2025-05-09|   2025-05-23|   2025-06-02|          TV|
|       2|         29|2025-05-01|   2025-05-24|   2025-06-19|      Laptop|
|       3|          6|2025-05-18|   2025-05-27|   2025-05-27|      Laptop|
|       4|         74|2025-05-06|   2025-06-01|   2025-06-07|Refrigerator|
|       5|         82|2025-04-30|   2025-06-11|   2025-06-11|          TV|
+--------+-----------+----------+-------------+-------------+------------+
only showing top 5 rows

📦 Customers Data
+-----------+-----------------+------+--------------------+------------+
|customer_id|    customer_name|region|               email|       phone|
+-----------+-----------------+------+--------------------+------------+
|          1|       Tara Walia|  West| aarav30@hot

In [0]:
from pyspark.sql.functions import col, when, count

# Join on customer_id
joined_df = orders_df.join(customers_df, on="customer_id", how="inner")
print("🔗 Joined Data:")
joined_df.show(5)


🔗 Joined Data:
+-----------+--------+----------+-------------+-------------+---------------+-----------------+------+--------------------+------------+
|customer_id|order_id|order_date|expected_date|delivery_date|        product|    customer_name|region|               email|       phone|
+-----------+--------+----------+-------------+-------------+---------------+-----------------+------+--------------------+------------+
|          2|      93|2025-04-22|   2025-05-26|   2025-06-13|             TV|      Arnav Dayal|  West|chandranfateh@sac...|910025040216|
|          3|      33|2025-05-14|   2025-05-31|   2025-06-10|   Refrigerator|Yuvaan Srinivasan| North|divyanshkhanna@ya...|  3968373591|
|          4|      10|2025-05-18|   2025-06-06|   2025-05-29|Washing Machine|        Anvi Gola|  East|devanalisha@venka...|  1558754080|
|          5|      21|2025-04-27|   2025-05-24|   2025-06-03|         Laptop|    Dishani Boase|  West|viswanathanindran...|915138533444|
|          6|      99|2025

In [0]:
# Add delay flag: delivery_date < expected_date
joined_df = joined_df.withColumn("is_delayed", when(col("delivery_date") < col("expected_date"), 1).otherwise(0))

# Preview delay flag
joined_df.select("customer_id", "order_id", "expected_date", "delivery_date", "is_delayed").show(5)


+-----------+--------+-------------+-------------+----------+
|customer_id|order_id|expected_date|delivery_date|is_delayed|
+-----------+--------+-------------+-------------+----------+
|         90|       1|   2025-05-23|   2025-06-02|         0|
|         29|       2|   2025-05-24|   2025-06-19|         0|
|          6|       3|   2025-05-27|   2025-05-27|         0|
|         74|       4|   2025-06-01|   2025-06-07|         0|
|         82|       5|   2025-06-11|   2025-06-11|         0|
+-----------+--------+-------------+-------------+----------+
only showing top 5 rows



In [0]:
# Group by region and count delays
delay_by_region = joined_df.groupBy("region").agg(
    count(when(col("is_delayed") == 1, True)).alias("delay_count")
)

# Show delay summary
print("📊 Delays by Region:")
delay_by_region.show()


📊 Delays by Region:
+------+-----------+
|region|delay_count|
+------+-----------+
| South|          9|
|  East|          9|
|  West|          8|
| North|          6|
+------+-----------+



In [0]:
# Save result to DBFS for download (optional)
delay_by_region.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/final_delays_by_region")


In [0]:
dbfs:/FileStore/final_delays_by_region/part-00000-tid-3735926507808247533-91957229-6d06-43c4-8288-22da2959187f-15-1-c000.csv